In [8]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

In [10]:
BASE_URL = "https://apidatos.ree.es/es/datos/"

HEADERS = {
    "accept": "application/json",
    "content-type": "application/json"
    # Se eliminó "host": "apidatos.ree.es" porque creo que no es obligatorio y me daba error en algun momento
}

ENDPOINTS = {
    "demanda": ("demanda/evolucion", "hour"),  
    "balance": ("balance/balance-electrico", "day"),
    "generacion": ("generacion/potencia-instalada", "month"), 
    "intercambios": ("intercambios/todas-fronteras-programados", "day")
}

In [12]:
# Función que consulta la API y devuelve los datos procesados
def get_data(endpoint_name, endpoint_info, params):
    path, time_trunc = endpoint_info
    params["time_trunc"] = time_trunc
    url = BASE_URL + path

    print(f"Descargando {endpoint_name} → {params['start_date']} a {params['end_date']}")

    try:
        response = requests.get(url, headers=HEADERS, params=params)
        if response.status_code != 200:
            print(f"Error {response.status_code} al consultar {endpoint_name}")
            print(response.text[:150])  # solo una parte del error
            return []
        response_data = response.json()
    except Exception as e:
        print(f"Error al conectar con la API: {e}")
        return []

    data = []

    for item in response_data.get("included", []):
        attrs = item.get("attributes", {})
        category = attrs.get("title")

        if "content" in attrs:
            for sub in attrs["content"]:
                sub_attrs = sub.get("attributes", {})
                sub_cat = sub_attrs.get("title")
                for entry in sub_attrs.get("values", []):
                    entry["primary_category"] = category
                    entry["sub_category"] = sub_cat
                    data.append(entry)
        else:
            for entry in attrs.get("values", []):
                entry["primary_category"] = category
                entry["sub_category"] = None
                data.append(entry)

    return data

In [14]:
# Función que recorre cada mes desde un año inicial hasta uno final y recopila los datos
def get_all_data_by_month(start_year=2019, end_year=2025):
    all_dfs = []

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            start = datetime(year, month, 1)
            if start > datetime.now():
                continue  # evita pedir datos futuros

            end = (start + timedelta(days=32)).replace(day=1) - timedelta(minutes=1)

            for name, (path, granularity) in ENDPOINTS.items():
                params = {
                    "start_date": start.strftime("%Y-%m-%dT%H:%M"),
                    "end_date": end.strftime("%Y-%m-%dT%H:%M"),
                    "geo_trunc": "electric_system",
                    "geo_limit": "peninsular",
                    "geo_ids": "8741"
                }

                month_data = get_data(name, (path, granularity), params)

                if month_data:
                    df = pd.DataFrame(month_data)
                    df["year"] = year
                    df["month"] = month
                    df["endpoint"] = name
                    all_dfs.append(df)

                time.sleep(1)  # pausa para no saturar la API

    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    else:
        return pd.DataFrame()


In [16]:
# Extracción
ree_data_df = get_all_data_by_month(2019, 2025)

Descargando demanda → 2019-01-01T00:00 a 2019-01-31T23:59
Descargando balance → 2019-01-01T00:00 a 2019-01-31T23:59
Descargando generacion → 2019-01-01T00:00 a 2019-01-31T23:59
Descargando intercambios → 2019-01-01T00:00 a 2019-01-31T23:59
Descargando demanda → 2019-02-01T00:00 a 2019-02-28T23:59
Descargando balance → 2019-02-01T00:00 a 2019-02-28T23:59
Descargando generacion → 2019-02-01T00:00 a 2019-02-28T23:59
Descargando intercambios → 2019-02-01T00:00 a 2019-02-28T23:59
Descargando demanda → 2019-03-01T00:00 a 2019-03-31T23:59
Descargando balance → 2019-03-01T00:00 a 2019-03-31T23:59
Descargando generacion → 2019-03-01T00:00 a 2019-03-31T23:59
Descargando intercambios → 2019-03-01T00:00 a 2019-03-31T23:59
Descargando demanda → 2019-04-01T00:00 a 2019-04-30T23:59
Descargando balance → 2019-04-01T00:00 a 2019-04-30T23:59
Descargando generacion → 2019-04-01T00:00 a 2019-04-30T23:59
Descargando intercambios → 2019-04-01T00:00 a 2019-04-30T23:59
Descargando demanda → 2019-05-01T00:00 a

In [18]:
ree_data_df

,value,percentage,datetime,primary_category,sub_category,year,month,endpoint
0,23209.473,1.0,2019-01-01T00:00:00.000+01:00,Demanda,None,2019,1,demanda
1,22406.330,1.0,2019-01-01T01:00:00.000+01:00,Demanda,None,2019,1,demanda
2,21090.596,1.0,2019-01-01T02:00:00.000+01:00,Demanda,None,2019,1,demanda
3,19896.113,1.0,2019-01-01T03:00:00.000+01:00,Demanda,None,2019,1,demanda
4,19146.631,1.0,2019-01-01T04:00:00.000+01:00,Demanda,None,2019,1,demanda
...,...,...,...,...,...,...,...,...
126548,-102.000,1.0,2025-05-24T00:00:00.000+02:00,andorra,saldo,2025,5,intercambios
126549,-87.900,1.0,2025-05-25T00:00:00.000+02:00,andorra,saldo,2025,5,intercambios
126550,-171.800,1.0,2025-05-26T00:00:00.000+02:00,andorra,saldo,2025,5,intercambios
126551,-206.600,1.0,2025-05-27T00:00:00.000+02:00,andorra,saldo,2025,5,intercambios


In [22]:
# GUARDAR EN CSV
ree_data_df.to_csv("datos_REE_2019_2025.csv", index=False)
